# 기계학습기반 단어중의성 해소

-  NLTK의 고전 데이터셋 Senseval(‘interest’, ‘hard’, ‘serve’) 를 이용해, 주변 문맥을 TF-IDF로 벡터화하고 로지스틱 회귀로 의미 분류

### 핵심 흐름
- 의미 태깅 코퍼스
   → 문맥 텍스트 추출
   → 벡터화
   → 지도학습 분류
   → 평가/예측

In [1]:
# ============================================
# WSD (Machine Learning-based) – Senseval + scikit-learn 데모 (Fixed)
# ============================================

# 1) 라이브러리 불러오기

# 자연어처리 도구 NLTK 불러오기
import nltk

# NLTK에 포함된 Word Sense Disambiguation(WSD)용 데이터셋 senseval 불러오기
from nltk.corpus import senseval

# 학습용 데이터와 평가용 데이터를 나누는 함수
from sklearn.model_selection import train_test_split

# 텍스트를 숫자 벡터로 바꿔주는 도구 (TF-IDF 방식)
from sklearn.feature_extraction.text import TfidfVectorizer

# 분류기 모델: 로지스틱 회귀 (여기서는 단어 의미를 분류하는 데 사용)
from sklearn.linear_model import LogisticRegression

# 여러 단계를 묶어서 한 번에 실행할 수 있게 해주는 파이프라인 도구
from sklearn.pipeline import Pipeline

# 모델 평가 지표 (정확도, 분류 리포트)
from sklearn.metrics import accuracy_score, classification_report

# 수치 계산을 위한 라이브러리 (배열, 벡터 연산 등)
import numpy as np


In [2]:
# 2) 필요한 리소스 다운로드 (처음 한 번만 실행하면 됨)
nltk.download('senseval')   # 의미 태깅 코퍼스(senseval) 다운로드
nltk.download('punkt')      # 문장/단어 토큰화를 위한 데이터 다운로드

# 3) 데이터셋 불러오기: 'interest' 단어에 대한 의미 태깅 코퍼스
# senseval.instances('interest.pos') → 'interest'라는 단어가 들어간 문맥과 정답 의미(sense) 제공
instances = senseval.instances('interest.pos')

[nltk_data] Downloading package senseval to /root/nltk_data...
[nltk_data]   Unzipping corpora/senseval.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [3]:
def context_to_text(inst):
    """
    Senseval 인스턴스(inst)에서 문맥(context)을 문자열로 변환하는 함수.
    - inst.context에는 단어가 문자열 또는 (단어, 품사) 튜플로 섞여 있음
    - 품사 태그가 붙어 있으면 튜플에서 단어만 사용
    - 불필요한 공백/기호 제거 후 모두 소문자로 변환
    - 알파벳이 하나라도 포함된 경우만 단어로 인정
    - 결과: 단어들을 공백으로 연결한 하나의 문자열
    """
    toks = []  # 단어들을 저장할 리스트
    for w in inst.context:
        if isinstance(w, tuple):
            w = w[0]                 # (word, tag) → word만 사용
        if isinstance(w, str):
            w = w.strip().lower()    # 공백 제거 + 소문자 변환
            if w and any(ch.isalpha() for ch in w):  # 알파벳이 하나라도 있으면 유효한 단어
                toks.append(w)
    return " ".join(toks)            # 단어들을 공백으로 이어붙여 문자열 반환

# 4) 입력(X: 문맥 텍스트)과 정답(y: 단어 의미) 구성 + 빈 문서 제거
X_texts, y_labels = [], []
for inst in instances:
    text = context_to_text(inst)      # 문맥을 문자열로 변환
    if text:                          # 내용이 비어 있지 않다면
        X_texts.append(text)          # 입력 데이터(X)에 추가
        y_labels.append(inst.senses[0])  # 해당 인스턴스의 정답 의미(sense)를 라벨(y)에 추가

# 데이터셋 크기와 라벨 종류 확인
print(f"총 샘플 수(빈 문서 제거 후): {len(X_texts)}")
print(f"레이블 종류: {sorted(set(y_labels))}")

총 샘플 수(빈 문서 제거 후): 2368
레이블 종류: ['interest_1', 'interest_2', 'interest_3', 'interest_4', 'interest_5', 'interest_6']


In [4]:
# 5) 학습/검증 데이터 분할
# 전체 데이터를 학습용(80%)과 검증용(20%)으로 나눔
# stratify=y_labels → 레이블 비율을 학습/검증에 동일하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X_texts, y_labels,
    test_size=0.2,      # 20%를 검증용으로 사용
    random_state=42,    # 재실행해도 동일하게 섞이도록 고정
    stratify=y_labels   # 클래스 불균형 문제를 줄이기 위해 레이블 비율 유지
)

# 6) 파이프라인 구성
# 텍스트 벡터화(TF-IDF) + 분류기(Logistic Regression)를 묶어 한 번에 실행 가능하게 함
pipeline = Pipeline([
    # (1) TfidfVectorizer: 문장을 단어/문맥 특징으로 변환
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),       # 단어 단위(uni-gram) + 2개 단어 연속(bi-gram)까지 사용
        min_df=1,                 # 최소 1개 문서에만 나와도 포함 (희소 단어도 허용)
        max_df=0.95,              # 너무 많이 나오는 단어(95% 이상 등장)는 제외
        token_pattern=r'(?u)\b[a-zA-Z]+\b'  # 알파벳 단어만 추출 (숫자/특수문자 제외)
    )),

    # (2) LogisticRegression: 지도학습 분류기 (다중 클래스 분류 가능)
    ("clf", LogisticRegression(max_iter=1000, n_jobs=None))
])

# 7) 학습 단계
# 학습 데이터(X_train, y_train)를 이용해 TF-IDF 벡터화 + 로지스틱 회귀 모델 학습
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, ngram_range=(1, 2),
                                 token_pattern='(?u)\\b[a-zA-Z]+\\b')),
                ('clf', LogisticRegression(max_iter=1000))])

In [5]:
# 8) 평가 단계

# 학습된 파이프라인을 사용해 검증용 데이터(X_test)의 예측 결과 생성
pred = pipeline.predict(X_test)

# 예측 결과와 실제 정답(y_test)을 비교하여 정확도(Accuracy) 계산
acc = accuracy_score(y_test, pred)

# 평가 결과 출력
print("📊 [모델 평가 결과]")        # 구분선과 함께 보기 좋게 출력
print(f"Accuracy: {acc:.3f}")       # 소수점 3자리까지 정확도 표시
print("-"*60)

# 분류 성능 보고서 출력
# 각 클래스(단어 의미)별 precision, recall, f1-score, 지원 데이터 개수(support)를 보여줌
# zero_division=0 → 특정 클래스가 예측되지 않아도 에러 대신 0으로 표시
print("세부 성능 보고서:")
print(classification_report(y_test, pred, zero_division=0))

📊 [모델 평가 결과]
Accuracy: 0.804
------------------------------------------------------------
세부 성능 보고서:
              precision    recall  f1-score   support

  interest_1       0.92      0.47      0.62        72
  interest_2       0.00      0.00      0.00         2
  interest_3       1.00      0.38      0.56        13
  interest_4       0.91      0.28      0.43        36
  interest_5       0.80      0.82      0.81       100
  interest_6       0.78      1.00      0.88       251

    accuracy                           0.80       474
   macro avg       0.74      0.49      0.55       474
weighted avg       0.82      0.80      0.78       474



In [6]:
# 9) 새 문장 예측 함수 정의
def predict_sense(sentence: str) -> str:
    """
    새로운 문장을 입력하면,
    학습된 파이프라인(pipeline)을 사용해 'interest' 단어의 의미를 예측.
    """
    # pipeline.predict는 리스트 형태 입력을 받으므로 [sentence]로 감싸줌
    # 결과는 배열 형태 → [0]으로 첫 번째(유일한) 결과만 반환
    return pipeline.predict([sentence])[0]

# 10) 데모 예측 테스트
# 실제 문장들을 넣어서 'interest' 단어가 어떤 의미로 분류되는지 확인
demo_sentences = [
    "There is growing interest among investors in the new fund.",   # 투자/재정적 관심
    "He paid interest on the loan for ten years.",                  # 대출 이자
    "Her main interest lies in classical music.",                   # 취미/관심사
]

print("🔎 [데모 문장 예측 결과]")
for s in demo_sentences:
    print(f"문장: {s}")                               # 입력 문장 출력
    print(f" → 예측된 의미(sense): {predict_sense(s)}")  # 예측된 의미 출력
    print("-"*50)                                   # 구분선 출력

🔎 [데모 문장 예측 결과]
문장: There is growing interest among investors in the new fund.
 → 예측된 의미(sense): interest_1
--------------------------------------------------
문장: He paid interest on the loan for ten years.
 → 예측된 의미(sense): interest_6
--------------------------------------------------
문장: Her main interest lies in classical music.
 → 예측된 의미(sense): interest_6
--------------------------------------------------


In [7]:
# 11) 클래스별 중요한 특징 단어 Top-5 확인

# pipeline에서 학습된 분류기(Logistic Regression) 객체 꺼내오기
clf = pipeline.named_steps["clf"]

# pipeline에서 TF-IDF 벡터화기 객체 꺼내오기
tfidf = pipeline.named_steps["tfidf"]

# TF-IDF에서 사용된 전체 단어(feature) 목록 가져오기
feature_names = np.array(tfidf.get_feature_names_out())

# 분류 대상 클래스(= 'interest' 단어의 여러 의미들) 가져오기
classes = clf.classes_

print("💡 [각 의미를 구분하는 중요한 단어 Top 5]")

# 각 의미 클래스별로 반복
for idx, cls in enumerate(classes):
    # clf.coef_[idx]: 해당 클래스와 관련된 가중치 벡터
    # np.argsort(...) → 큰 값 순으로 정렬
    # [::-1][:5] → 상위 5개 단어 인덱스 선택
    top_idx = np.argsort(clf.coef_[idx])[::-1][:5]

    # 선택된 단어(feature_names[top_idx]) 출력
    print(f"- {cls}: {', '.join(feature_names[top_idx])}")

💡 [각 의미를 구분하는 중요한 단어 Top 5]
- interest_1: interest in, buying, buying interest, investor interest, investor
- interest_2: might, baseball, earthquake, florida, of interest
- interest_3: pursue, to pursue, pursue other, interests, resigned
- interest_4: interests, best, interests of, not, public
- interest_5: interests, short interest, interests in, short, in
- interest_6: rates, interest rates, rate, bonds, interest rate
